# Phenogenetic architectures

This is where things start to get interesting. The ability to construct
and modify phenogenetic architectures with arbitrary complexity was the
core motivating feature behind `xftsim`. In v0.9 architectures are
expressed in a small lavaan-style formula DSL parsed by
`xftsim.parser`, replacing the v0.3 hierarchy of
`ArchitectureComponent` classes (`AdditiveGeneticComponent`,
`LinearTransformationComponent`, `LinearVerticalComponent`,
`ProductComponent`, `SumAllTransformation`, …).

The DSL has two kinds of right-hand sides:

| RHS form | Meaning |
| --- | --- |
| `function(args)` | A *generative* or *reference* component (genetic, noise, threshold, mother/father/parent, sibling_*) |
| arithmetic expression | An *aggregation* — sums, products, scalar mixing of already-computed components |

Optional `| GROUPING` suffix on a function call applies grouping
(e.g. `| FID` shares a draw across siblings).

The four kinds of formula nodes:

| DSL function | Component class | Purpose |
| --- | --- | --- |
| `genetic(eff)` | `GeneticComponent` | Additive genetic component G = X β |
| `mvGenetic(eff)` | `MVGeneticComponent` | Multivariate genetic component for tuple LHS `(a, b)` |
| `haplotypeGenetic(eff, haplotype='maternal')` | `HaplotypeGeneticComponent` | One-haplotype genetic, for indirect-genetic-effect models |
| `noise(σ²)` | `NoiseComponent` | iid N(0, σ²) noise |
| `cnoise(cov=[[…]])` | `CNoiseComponent` | Multivariate correlated N(0, Σ) noise — tuple LHS |
| `threshold(source, value)` | `ThresholdComponent` | Binarize `source > value` (liability-threshold model) |
| `mother(pheno, …)` | `MotherComponent` | Look up mother's phenotype from previous generation |
| `father(pheno, …)` | `FatherComponent` | Look up father's phenotype |
| `parent(pheno, …)` | `ParentComponent` | Mid-parent average |
| `sibling_mean(s)`, `sibling_sum(s)`, `sibling_count(s)`, `sibling_any(s)`, `sibling_eldest(s)`, `sibling_youngest(s)` | sibling components | Aggregate `s` within FID |
| (arithmetic) | `AggregationComponent` | `+`, `-`, `*`, `/`, scalar literals — sums, GxE products, anything algebraic |


In [ ]:
import numpy as np
import xftsim as xft

np.random.seed(123)
xft.config.print_durations_threshold = 10.

# A small founder population reused throughout the tutorial.
N, M = 800, 200
hap = xft.founders.founder_haplotypes_uniform_AFs(n=N, m=M)
rmap = xft.reproduce.RecombinationMap.from_haplotypes(hap, p=0.1)
mating = xft.mate.RandomMating(offspring_per_pair=2)


## The minimal additive-genetic architecture

The widely-used additive-genetic-plus-noise model

$$y = X\beta + e, \quad e \sim N(0, \sigma_e^2), \quad \beta \stackrel{iid}{\sim} N(0, \sigma_\beta^2)$$

is expressed as three formula lines: one for `Y.G` (genetic), one for
`Y.E` (noise), and one aggregation `Y` for the outcome:


In [ ]:
eff = xft.effect.AdditiveEffects.from_h2(h2=0.5, m=M, seed=1)

arch = xft.arch.Architecture(
    formula='''
    Y.G ~ genetic(eff)
    Y.E ~ noise(0.5)
    Y   ~ Y.G + Y.E
    ''',
    effects={'eff': eff},
)
arch


There are three nodes in the architecture's DAG (one per LHS):


In [ ]:
len(arch.nodes)


In [ ]:
for node in arch.nodes:
    print(node)


We can run a small simulation against this architecture:


In [ ]:
sim = xft.sim.Simulation(
    founder_haplotypes=hap,
    architecture=arch,
    recombination_map=rmap,
    mating_regime=mating,
    statistics=[xft.stats.SampleStatistics()],
    seed=42,
)
sim.run(n_generations=1)
ss = sim.results[-1].statistics['SampleStatistics']
list(zip(ss['keys'], np.diag(ss['cov'])))


## Multi-trait architectures

For multiple independent traits, just add more groups of three
lines. Effects are typically drawn from `AdditiveEffects.from_h2(...)`
once per trait (separate effects → orthogonal genetic effects, i.e.
zero pleiotropy):


In [ ]:
eff_height = xft.effect.AdditiveEffects.from_h2(h2=0.6, m=M, seed=1)
eff_bmd    = xft.effect.AdditiveEffects.from_h2(h2=0.4, m=M, seed=2)

arch_2trait = xft.arch.Architecture(
    formula='''
    height.G ~ genetic(eff_h)
    height.E ~ noise(0.4)
    height   ~ height.G + height.E

    BMD.G ~ genetic(eff_b)
    BMD.E ~ noise(0.6)
    BMD   ~ BMD.G + BMD.E
    ''',
    effects={'eff_h': eff_height, 'eff_b': eff_bmd},
)


### Correlated genetic effects

For correlated genetic effects across `k` traits, use a
`MultivariateEffects` object with a tuple LHS and the `mvGenetic`
builtin:


In [ ]:
eff_mv = xft.effect.MultivariateEffects.from_h2_rg(
    h2=[0.5, 0.4], rg=0.25, m=M, seed=3,
)

arch_mv = xft.arch.Architecture(
    formula='''
    (height.G, BMD.G) ~ mvGenetic(eff_mv)
    height.E ~ noise(0.5)
    BMD.E    ~ noise(0.6)
    height   ~ height.G + height.E
    BMD      ~ BMD.G    + BMD.E
    ''',
    effects={'eff_mv': eff_mv},
)


## Noise components

`noise(σ²)` draws iid N(0, σ²). For correlated multivariate noise use
`cnoise(cov=[[...]])` with a tuple LHS:


In [ ]:
arch_cnoise = xft.arch.Architecture(
    formula='''
    (height.E, BMD.E) ~ cnoise(cov=[[0.5, 0.15], [0.15, 0.6]])
    '''
)
arch_cnoise.nodes


The covariance matrix dimension must match the number of LHS names.

### Grouped noise

Adding `| FID` (or `| sex`, or `| mother`, …) to a `noise(...)` or
`cnoise(...)` call draws **one shared value per group** and broadcasts
it to all individuals in that group. This is the easiest way to model
shared-environment effects within families:


In [ ]:
arch_shared = xft.arch.Architecture(
    formula='''
    Y.G       ~ genetic(eff)
    Y.shared  ~ noise(0.2) | FID
    Y.E       ~ noise(0.3)
    Y         ~ Y.G + Y.shared + Y.E
    ''',
    effects={'eff': eff},
)


## Causal dependence (linear transformations)

Suppose income depends linearly on years of education:

$$\text{Edu}_E \sim N(0, 1), \quad \text{Income}_E \sim N(0, 0.5),$$
$$\text{Edu} = \text{Edu}_E, \quad \text{Income} = \text{Income}_E + \sqrt{0.5} \cdot \text{Edu}_E.$$

In the legacy interface this required a `LinearTransformationComponent`
plus a `SumAllTransformation`. In v0.9 the entire model is one
arithmetic line per outcome:


In [ ]:
arch_edu_income = xft.arch.Architecture(
    formula='''
    edu.E    ~ noise(1.0)
    income.E ~ noise(0.5)
    edu      ~ edu.E
    income   ~ income.E + 0.7071 * edu.E
    '''
)
arch_edu_income


Multivariate causal dependence is just additional terms in the
aggregation expression:


In [ ]:
arch_edu_opp_income = xft.arch.Architecture(
    formula='''
    edu.E        ~ noise(1.0)
    opp.E        ~ noise(1.0)
    income.E     ~ noise(0.5)
    edu          ~ edu.E
    opp          ~ opp.E
    income       ~ income.E + 0.5 * edu.E + 0.5 * opp.E
    '''
)


## Vertical transmission

Vertical transmission is a causal dependence *across generations* — a
component on the proband depends on a parent's phenotype from the
previous generation. The DSL primitives are `mother(...)`,
`father(...)`, and `parent(...)` (mid-parent average).

For example, half of wealth is individual-specific noise and the rest
is split between mother and father:

$$W_E \sim N(0, 0.5), \quad W = W_E + 0.5 \cdot W^{(m)} + 0.5 \cdot W^{(f)}.$$


In [ ]:
arch_vt = xft.arch.Architecture(
    formula='''
    wealth.E   ~ noise(0.5)
    wealth.VTm ~ mother(wealth, founder=noise(1.0))
    wealth.VTf ~ father(wealth, founder=noise(1.0))
    wealth     ~ wealth.E + 0.5 * wealth.VTm + 0.5 * wealth.VTf
    '''
)


At **generation 0**, `mother(...)`/`father(...)` can't look up a
phenotype because there are no parents. The `founder=noise(σ²)` kwarg
supplies the founder-generation initialisation — a draw from
`N(0, σ²)` per individual. Without it, the component would be zero at
gen 0.

The `normalize=True` kwarg (matching the legacy
`LinearVerticalComponent(normalize=True)` behaviour) standardises the
parental values *after* the pedigree lookup. Under assortative mating
this is what prevents the transmitted component's variance from
growing unboundedly across generations:


In [ ]:
# Stable-variance VT for K traits at fraction θ of phenotypic variance.
# (One trait shown here; the K=5 / θ=0.05 version appears in the manuscript
# Figure 3 reproduction tests.)
import numpy as np
K = 1
theta = 0.05
coef = float(np.sqrt(theta / (2 * K)))   # per-parent, per-trait coefficient

arch_vt_norm = xft.arch.Architecture(
    formula=f'''
    Y.G    ~ genetic(eff)
    Y.E    ~ noise(0.45)
    Y.VTm  ~ mother(Y, normalize=True, founder=noise(1.0))
    Y.VTf  ~ father(Y, normalize=True, founder=noise(1.0))
    Y      ~ Y.G + Y.E + {coef} * Y.VTm + {coef} * Y.VTf
    ''',
    effects={'eff': eff},
)


## Gene-by-environment interactions

The most common GxE model is

$$Y = G + E + \alpha \cdot G \cdot E + \varepsilon$$

with $\alpha = \sqrt{\sigma_{GxE}^2 / (\sigma_G^2 \cdot \sigma_E^2)}$.

In v0.9 this is one line — the aggregation evaluator supports `*` as
pairwise multiplication of two phenotype arrays:


In [ ]:
vg, ve, veps, vgxe = 0.3, 0.3, 0.3, 0.1
alpha = float(np.sqrt(vgxe / (vg * ve)))
eff_gxe = xft.effect.AdditiveEffects.from_h2(h2=vg, m=M, seed=4)

arch_gxe = xft.arch.Architecture(
    formula=f'''
    Y.G   ~ genetic(eff_gxe)
    Y.E   ~ noise({ve})
    Y.eps ~ noise({veps})
    Y     ~ Y.G + Y.E + Y.eps + {alpha} * Y.G * Y.E
    ''',
    effects={'eff_gxe': eff_gxe},
)


Higher-order interactions are just longer expressions:


In [ ]:
arch_gxexeps = xft.arch.Architecture(
    formula=f'''
    Y.G   ~ genetic(eff_gxe)
    Y.E   ~ noise({ve})
    Y.eps ~ noise({veps})
    Y     ~ Y.G + Y.E + Y.eps + {alpha} * Y.G * Y.E + 0.1 * Y.G * Y.E * Y.eps
    ''',
    effects={'eff_gxe': eff_gxe},
)


## Liability-threshold binarisation

For binary outcomes (e.g. disease diagnoses) under the
liability-threshold model: compute a continuous liability, then apply
`threshold(source, value)` to indicator-binarise it. For prevalence
`K`, the threshold is the standard-normal upper-`K` quantile:


In [ ]:
from scipy.stats import norm

prev = 0.05
thresh = float(norm.ppf(1 - prev))
eff_lia = xft.effect.AdditiveEffects.from_h2(h2=0.5, m=M, seed=5)

arch_dx = xft.arch.Architecture(
    formula=f'''
    DX.G  ~ genetic(eff_lia)
    DX.E  ~ noise(0.5)
    DX    ~ DX.G + DX.E
    DX.dx ~ threshold(DX, {thresh})
    ''',
    effects={'eff_lia': eff_lia},
)


Mating regimes work on the *continuous* phenotype (`DX`) — the binary
diagnosis (`DX.dx`) is just an additional component for downstream
statistics.

## Sibling aggregation

The `sibling_*` builtins aggregate a source phenotype within the
group defined by either an explicit `| GROUPING` or — by default —
`| FID`. They produce one value per group, broadcast back to all
members. Useful for things like sibship size, family-mean phenotype,
or eldest/youngest indicators.


In [ ]:
arch_sibs = xft.arch.Architecture(
    formula='''
    Y.G       ~ genetic(eff)
    Y.E       ~ noise(0.5)
    Y         ~ Y.G + Y.E
    Y.fammean ~ sibling_mean(Y)
    Y.sibcount ~ sibling_count(Y)
    ''',
    effects={'eff': eff},
)


Available sibling aggregators: `sibling_mean`, `sibling_sum`,
`sibling_any`, `sibling_count`, `sibling_eldest`, `sibling_youngest`.

## Programmatic construction

Formula strings are the recommended way to build an `Architecture`,
but the constructor is also available programmatically (and the
formula parser produces the same `ArchNode`s under the hood):


In [ ]:
arch_prog = xft.arch.Architecture()
arch_prog.add('Y.G', xft.arch.GeneticComponent(eff))
arch_prog.add('Y.E', xft.arch.NoiseComponent(0.5))
arch_prog.add('Y',   xft.arch.AggregationComponent('Y.G + Y.E'))
arch_prog


## Validation: cycles and unresolved references

The DAG is topologically sorted on construction (or on the first
`.nodes` access). Cycles raise immediately:


In [ ]:
try:
    bad = xft.arch.Architecture(formula='''
    A ~ B + 1
    B ~ A + 1
    ''')
except ValueError as e:
    print('caught:', e)


Unresolved input names also raise immediately:


In [ ]:
try:
    bad = xft.arch.Architecture(formula='''
    Y ~ Y.G + Y.E
    ''')
except ValueError as e:
    print('caught:', e)


This kind of structural error used to surface only at simulation time
in the legacy `SumAllTransformation` interface, which made
architectures with circular dependences hard to debug. The formula
parser catches it up front.

## What's gone vs the legacy interface

A short cheat-sheet of API changes for porters of legacy code:

| Legacy | v0.9 |
| --- | --- |
| `arch.GCTA_Architecture(h2=[...], phenotype_name=[...], haplotypes=...)` | formula DSL + `effect.AdditiveEffects.from_h2` |
| `arch.AdditiveGeneticComponent(effects)` | `genetic(eff)` |
| `arch.AdditiveNoiseComponent(variances=[...])` | one `noise(σ²)` per LHS |
| `arch.CorrelatedNoiseComponent(vcov=...)` | `cnoise(cov=[[...]])` with tuple LHS |
| `arch.LinearTransformationComponent(input_cindex, output_cindex, coefficient_matrix, normalize)` | arithmetic expression with scalar coefficients |
| `arch.LinearVerticalComponent(...)` | `mother(...)` / `father(...)` / `parent(...)`, with `normalize=True` and `founder=noise(σ²)` |
| `arch.ProductComponent(input_cindex, ...)` | `*` in an aggregation expression |
| `arch.SumAllTransformation(input_cindex)` | aggregation expression — typically just `~` plus a sum |
| `Architecture.draw_dependency_graph()` | removed — inspect `arch.nodes` directly or use `print(arch)` |
| `xft.sim.DemoSimulation('BGRM')` | removed — build a `Simulation` explicitly (see [`xftsim/quickstart.py`](https://github.com/border-lab/xftsim/blob/v0.9alpha/xftsim/quickstart.py)) |
